이 실습에서는 scikit-learn에서 제공하는 Breast Cancer Wisconsin 데이터셋을 사용합니다.

- 목적: 종양이 '악성(malignant)'인지 '양성(benign)'인지 분류하는 이진 분류 문제
- 입력 데이터(X): 종양의 크기, 형태, 질감 등과 관련된 여러 개의 수치형 특성(feature)
- 타깃(y):
    - 0: 악성(malignant)
    - 1: 양성(benign)

목표
1) 동일한 데이터에 대해 3가지 모델을 학습한다.
   - Random Forest (배깅)
   - Gradient Boosting (부스팅)
   - Logistic Regression (선형 분류)
2) Logistic Regression에만 표준화를 적용한다.
   - 로지스틱 회귀는 feature 스케일에 민감하므로 표준화(StandardScaler)가 유의미
3) 각 모델의 예측 결과를 classification_report로 비교하고, confusion_matrix로 오분류 패턴을 확인한다.


In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# 1. 데이터 로드
# scikit-learn 내장 데이터셋을 로드합니다.
data = load_breast_cancer()

In [3]:
# 2. 독립 변수(X)와 종속 변수(y) 설정
# feature는 DataFrame 형태, target은 1차원 라벨 배열/Series로 구성합니다.
X = pd.DataFrame(data.data, columns=data.feature_names)     # TODO: feature 데이터로 X 구성
y = data.target     # TODO: target 라벨로 y 구성 (0/1)

In [4]:
# 3. 데이터 분리 (훈련 80%, 테스트 20%)
# random_state를 고정해 결과를 재현 가능하게 합니다.
X_train, X_test, y_train, y_test = train_test_split(
    X,     # TODO
    y,     # TODO
    test_size=0.2,
    random_state=42,
    stratify=y      # TODO: 클래스 비율 유지
)

In [5]:
# 4. 데이터 스케일링 (로지스틱 회귀에 적용)
# 로지스틱 회귀는 스케일에 민감할 수 있으므로 표준화를 적용합니다.
# 주의: 훈련 데이터로만 scaler 기준을 학습(fit)하고, 테스트 데이터는 transform만 수행해야 합니다.
scaler = StandardScaler()                 # TODO: StandardScaler 생성
X_train_scaled = scaler.fit_transform(X_train)         # TODO: 훈련 데이터 표준화 (fit 포함)
X_test_scaled = scaler.transform(X_test)          # TODO: 테스트 데이터 표준화 (transform만)


In [6]:
# 5. 모델 설정
# (1) Random Forest - 배깅 방식
# 배깅(bagging) 방식의 랜덤 포레스트 모델을 생성합니다. 
# - n_estimators=10: 트리의 개수를 10개로 설정
# - random_state=42: 결과 재현을 위해 고정된 난수 시드를 설정
rf_model = RandomForestClassifier(n_estimators=10, random_state=42)  # TODO: RandomForestClassifier 생성 (예: n_estimators, random_state 설정)

# (2) Gradient Boosting - 부스팅 방식
'''
Gradient Boosting은 여러 개의 약한 학습기(주로 결정 트리)를 순차적으로 학습하여 예측 성능을 향상시키는 앙상블 학습 방법 중 하나입니다. 
각 단계에서 이전 모델이 잘못 예측한 데이터를 더 잘 맞추도록 가중치를 부여하여 새로운 모델을 학습시킵니다. 
최종적으로 모든 약한 학습기의 예측 결과를 합산하여 강력한 예측 모델을 생성합니다.

GradientBoostingClassifier는 이 아이디어를 기반으로 한 분류 모델입니다. 
모델은 먼저 예측이 틀린 데이터에 더 많은 비중을 두고 학습하며, 이를 반복해 나가면서 오차를 줄입니다. 
이 과정은 손실 함수의 기울기를 최소화하는 방식으로 진행되기 때문에 'Gradient'라는 용어가 사용됩니다. 
일반적으로, 과적합을 방지하기 위해 트리의 깊이나 학습률(learning rate) 등을 조절할 수 있습니다.
'''
# 부스팅 방식의 Gradient Boosting 모델을 생성합니다.
# - n_estimators=10: 트리 개수를 10개로 설정
# - learning_rate=0.1: 학습률을 0.1로 설정하여 학습 속도 조절
# - max_depth=3: 각 트리의 최대 깊이를 3으로 설정하여 과적합 방지
gb_model = GradientBoostingClassifier(n_estimators=10, learning_rate=0.1,max_depth=3)  # TODO: GradientBoostingClassifier 생성 (예: n_estimators, learning_rate, max_depth, random_state 설정)

# (3) Logistic Regression - 로지스틱 회귀 모델
# Logistic Regression 모델을 생성하며 L2 정규화를 사용합니다.
# - penalty='l2': L2 정규화(릿지 회귀)를 사용
# - C=1.0: 정규화 강도 조절, C 값이 클수록 약한 정규화, 작을수록 강한 정규화
lr_model = LogisticRegression(penalty='l2', C=1.0, random_state=42)  # TODO: LogisticRegression 생성 (예: penalty='l2', C, random_state 등)


In [7]:
models = {
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
    "Logistic Regression": lr_model
}

In [8]:
# 6. 모델 학습 및 평가
# - Logistic Regression은 표준화된 데이터를 사용
# - 트리 기반 모델은 원본 데이터를 사용
for model_name, model in models.items():
    if model_name == "Logistic Regression":
        # TODO: 로지스틱 회귀 학습 (스케일된 훈련 데이터 사용)
        model.fit(X_train_scaled, y_train)
        # TODO: 로지스틱 회귀 예측 (스케일된 테스트 데이터 사용)
        y_pred = model.predict(X_test_scaled)
    else:
        # TODO: 트리 기반 모델 학습 (원본 훈련 데이터 사용)
        model.fit(X_train_scaled, y_train)
        # TODO: 트리 기반 모델 예측 (원본 테스트 데이터 사용)
        y_pred = model.predict(X_test_scaled)

    # 7. 성능 평가 출력
    # classification_report는 precision/recall/f1-score 등을 클래스별로 요약합니다.
    report = classification_report(y_test,y_pred)  # TODO: classification_report 생성

    # confusion_matrix는 실제 라벨과 예측 라벨의 조합을 표로 보여줍니다.
    cm = confusion_matrix(y_test,y_pred)      # TODO: confusion_matrix 생성

    print(f"\n==== {model_name} ====")
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(report)


==== Random Forest ====
Confusion Matrix:
[[39  3]
 [ 4 68]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92        42
           1       0.96      0.94      0.95        72

    accuracy                           0.94       114
   macro avg       0.93      0.94      0.93       114
weighted avg       0.94      0.94      0.94       114


==== Gradient Boosting ====
Confusion Matrix:
[[37  5]
 [ 3 69]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90        42
           1       0.93      0.96      0.95        72

    accuracy                           0.93       114
   macro avg       0.93      0.92      0.92       114
weighted avg       0.93      0.93      0.93       114


==== Logistic Regression ====
Confusion Matrix:
[[41  1]
 [ 1 71]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98

c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
